In [ ]:
from platform import python_version
print(python_version())

In [ ]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"
ROOT_COLAB = ROOT0 / "colab"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import pdreadcsv, pdwritecsv, create_dir
from libs.MTD_lib import *
from libs.dashcyto_lib import DASH_CYTO
from libs.calc_degs_lib import CALC_DEGS

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

In [ ]:
root0 = Path(dic_yml['root0'])
root0_data = Path(dic_yml['root0_data'])
root_colab = root0 / 'colab'

email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"


root_project = root0_data / s_project
PSI_ID = 'TCGA-BRCA'
PSI_ID = 'TCGA-ACC'
PSI_ID = 'TCGA-CESC'

disease = PSI_ID

root_project = create_dir(root0_data, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']

case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=root0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

In [ ]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, root0=root0, root0_data=root0_data,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(mtd.echo_parameters())

In [ ]:
mtd.geneset_num, mtd.geneset_lib

In [ ]:
mtd.gene.df_my_gene.head(2)

### Studying default non-conding genes

In [ ]:
ret, degs, degs_ensembl, dfdegs = mtd.open_case_params(case, LFC_cut=1, lfc_FDR_cut=0.05, verbose=False)
print("\nEcho Parameters - default:")
print(mtd.echo_parameters())


In [ ]:
dflfc = mtd.dflfc
dflfc.biotype.unique()

### Non-codings

http://www.rnanut.net/lncrnadisease/index.php/home/info/download

### Calculating DEGs statistics

### For each LFC/FDR cutoff set we get diferent set of DEGs
  - LFC: LFC cutoff and FDR_LFC cutoff
  - Pathway: fdr and pval pathway cutoff and min num of genes

### Up and Down
  - Up and Down DEGs/DAPs
  - Up and Down in pathways

### there are 2 statistical tables
  - pval/fdr cutoff x degs
  - pval/fdr/geneset/quantile degs_in_pathway, num_pathways

### Biotypes

https://www.ensembl.org/info/genome/genebuild/biotypes.html

### Ensembl Gallery

https://www.ensembl.org/info/website/gallery.html

In [ ]:
lfc_cut = 1.0
fdr_cut = 0.05

verbose=False
force=True

dfnc, msg = mtd.filter_non_coding(lfc_cut=lfc_cut, fdr_cut=fdr_cut, verbose=verbose, force=force)
print(msg, '\n')
dfnc.head(6)

In [ ]:
biotype_list = ['lncRNA', 'transcribed_unitary_pseudogene',
       'unprocessed_pseudogene', 
       'transcribed_unprocessed_pseudogene', 'processed_pseudogene',
       'snoRNA', 'miRNA',
       'polymorphic_pseudogene', 'transcribed_processed_pseudogene',
       'snRNA', 'misc_RNA', 'IG_V_pseudogene',
       'unitary_pseudogene', 'rRNA_pseudogene',
       'translated_unprocessed_pseudogene']


dflfc_noncod = mtd.dflfc_ori[ mtd.dflfc_ori.biotype.isin(biotype_list)].copy()
dflfc_noncod = dflfc_noncod.sort_values(["biotype", "symbol"])
dflfc_noncod.reset_index(drop=True, inplace=True)

uniq_biotypes = np.unique(dflfc_noncod.biotype)
print(f"len noncoding: {len(dflfc_noncod)}/{len(mtd.dflfc_ori)}, unique biotypes: {uniq_biotypes}")

# fname = f'non_codings_for_{mtd.disease}.tsv'
# pdwritecsv(dflfc_noncod, fname, mtd.root_nc, verbose=True)
dflfc_noncod.head()

In [ ]:
cols = ['LNC', 'protein_coding', 'hasSymbol', 'UNK']

dflfc_noncod = dflfc[ ~dflfc.biotype.isin(cols)]
print(len(dflfc_noncod), dflfc_noncod.biotype.unique())

fname = f'non_codings_degs_for_{mtd.case}.tsv'
# pdwritecsv(dflfc_noncod, fname, mtd.root_result, verbose=True)

dflfc_noncod.head()

### LncRNA_Disease v3.0

http://www.rnanut.net/lncrnadisease/index.php/home/info/download

http://www.rnanut.net/lncrnadisease/

### miR2Disease

http://watson.compbio.iupui.edu:8080/miR2Disease/index.jsp

### Disease-associated Enhancer Catalog

http://biocc.hrbmu.edu.cn/DiseaseEnhancer/

In [ ]:
root_lrd = create_dir(mtd.root_colab, 'lncRNA')
fname = 'all_ncRNA_disease_information.tsv'

fullname = os.path.join(root_lrd, fname)

if os.path.exists(fullname):
    dfdis = pdreadcsv(fname, root_lrd)
    print(len(dfdis))
else:
    dfdis = pd.DataFrame()
    
dfdis.head(3)

In [ ]:
symbols = dflfc_noncod.symbol
print("dflfc_noncod", len(symbols))
"; ".join(symbols)

In [ ]:
dfdis['ncRNA_Symbol'].to_list()[:5]

In [ ]:
symbols_out = [x for x in symbols if x not in dfdis['ncRNA_Symbol'].to_list()]
print(len(symbols_out))
"; ".join(symbols_out)

In [ ]:
dfin = dfdis[dfdis['ncRNA_Symbol'].isin(symbols)]
print("dflfc_noncod symbols", len(symbols))
print("dfdis", len(dfdis))
print("df commons", len(dfin))
print("df commons unique", len(dfin['ncRNA_Symbol'].unique()))
dfin

In [ ]:
symb_identified_list = dfin['ncRNA_Symbol'].unique()
symb_identified_list[:10]

In [ ]:
fname = f"lnc_disease_identification_for_{mtd.disease}.tsv"
pdwritecsv(dfin, fname, mtd.root_nc, verbose=True)

In [ ]:
print(len(dfin))
dfin[ dfin['ncRNA_Symbol'] == symb_identified_list[0] ]

In [ ]:
dfin[ dfin['ncRNA_Symbol'] == symb_identified_list[1] ]

In [ ]:
dflfc_LFC = dflfc[ dflfc.biotype == 'LNC']

fname = f'LNC_special_non_codings_degs_for_{mtd.case}.tsv'
# pdwritecsv(dflfc_LFC, fname, mtd.root_result, verbose=True)

print(len(dflfc_LFC))
dflfc_LFC.head()

In [ ]:
symbols = dflfc_LFC.symbol.unique()
len(symbols), ", ".join(symbols)

In [ ]:
df_ident_LNC = dfdis[dfdis['ncRNA_Symbol'].isin(symbols)]
symb_identified_list = df_ident_LNC['ncRNA_Symbol'].unique()
symb_identified_list